# 0. Скачивание и обработка

Выгрузка данных из Wikidata SPARQL endpoint для открытых библиотек Python экосистемы PyTorch и их локальное сохранение.

In [1]:
import requests
import json
import os

url = "https://query.wikidata.org/sparql"
headers = {"User-Agent": "DomainVerification/1.0 (test@example.org)"}

query_entities = """
SELECT DISTINCT ?item ?itemLabel ?repo WHERE {
  ?item wdt:P31 wd:Q29642950 ;
        wdt:P1547 wd:Q47509047 ;
        wdt:P1324 ?repo .
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en,ru,mul,da". }
}
ORDER BY ?itemLabel
"""

response = requests.get(url, params={"query": query_entities, "format": "json"}, headers=headers)
data = response.json()
bindings = data["results"]["bindings"]

entities_dict = {}
for b in bindings:
    qid = b["item"]["value"].split("/")[-1]
    name = b["itemLabel"]["value"]
    repo = b["repo"]["value"]
    if name.startswith("Q") and name[1:].isdigit() and repo:
        name = repo.rstrip("/").split("/")[-1]
    if qid not in entities_dict:
        entities_dict[qid] = {
            "qid": qid,
            "name": name,
            "repo": repo
        }
entities = list(entities_dict.values())

os.makedirs("data", exist_ok=True)
filepath = "data/task1_entities.json"
with open(filepath, "w", encoding="utf-8") as f:
    json.dump(entities, f, ensure_ascii=False, indent=2)

file_size = os.path.getsize(filepath)
with open(filepath, "r", encoding="utf-8") as f:
    content = f.read()

print("Число выгруженных уникальных сущностей:", len(entities))
print("Файл сохранен:", filepath)
print("Размер файла в байтах:", file_size)
print("Число строк в файле:", len(content.splitlines()))

print("\nПервые 15 строк сохраненного файла:")
for line in content.splitlines()[:15]:
    print(line)

Число выгруженных уникальных сущностей: 52
Файл сохранен: data/task1_entities.json
Размер файла в байтах: 6174
Число строк в файле: 262

Первые 15 строк сохраненного файла:
[
  {
    "qid": "Q96631404",
    "name": "PyKEEN",
    "repo": "https://github.com/pykeen/pykeen"
  },
  {
    "qid": "Q85847011",
    "name": "PyTorch Lightning",
    "repo": "https://github.com/Lightning-AI/lightning"
  },
  {
    "qid": "Q116908797",
    "name": "STonKGs",
    "repo": "https://github.com/stonkgs/stonkgs"


# 1. Свойства домена

## 1.1. Все свойства всех сущностей домена

In [2]:
query_all_props = """
SELECT DISTINCT ?item ?itemLabel ?p ?repo WHERE {
  ?item wdt:P31 wd:Q29642950 ;
        wdt:P1547 wd:Q47509047 ;
        wdt:P1324 ?repo ;
        ?p ?val .
  FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en,ru,mul,da". }
}
"""

resp_all_props = requests.get(url, params={"query": query_all_props, "format": "json"}, headers=headers)
all_prop_rows = resp_all_props.json()["results"]["bindings"]

entity_props = {}
all_unique_pids = set()

for r in all_prop_rows:
    qid = r["item"]["value"].split("/")[-1]
    name = r["itemLabel"]["value"]
    repo = r["repo"]["value"]
    if name.startswith("Q") and name[1:].isdigit() and repo:
        name = repo.rstrip("/").split("/")[-1]
    pid = r["p"]["value"].split("/")[-1]
    all_unique_pids.add(pid)
    if qid not in entity_props:
        entity_props[qid] = {"name": name, "repo": repo, "props": set()}
    entity_props[qid]["props"].add(pid)

print("Всего сущностей в обработке:", len(entity_props))
print("Всего различных предикатов найдено в домене:", len(all_unique_pids))
print("Список всех уникальных предикатов:", sorted(list(all_unique_pids)))

Всего сущностей в обработке: 52
Всего различных предикатов найдено в домене: 22
Список всех уникальных предикатов: ['P1324', 'P1343', 'P1401', 'P1451', 'P1547', 'P178', 'P2037', 'P2078', 'P2671', 'P275', 'P277', 'P2888', 'P31', 'P3454', 'P348', 'P366', 'P4162', 'P4901', 'P5568', 'P6216', 'P6251', 'P856']


## 1.2. Распределение сущностей по числу свойств

In [3]:
counts = [len(data["props"]) for qid, data in entity_props.items()]
min_props = min(counts)
max_props = max(counts)
avg_props = sum(counts) / len(counts)

freq = {}
for c in sorted(counts):
    freq[c] = freq.get(c, 0) + 1

print("Минимальное число свойств у сущности:", min_props)
print("Максимальное число свойств у сущности:", max_props)
print("Среднее число свойств у сущности:", round(avg_props, 2))

print("\nРаспределение сущностей по числу свойств:")
print(f"{'Число свойств':15} | {'Число сущностей':15} | Доля от выборки")
print("-" * 50)
for prop_c, ent_c in sorted(freq.items()):
    share = (ent_c / len(entity_props)) * 100
    print(f"{prop_c:15} | {ent_c:15} | {share:6.2f}%")

Минимальное число свойств у сущности: 5
Максимальное число свойств у сущности: 17
Среднее число свойств у сущности: 7.85

Распределение сущностей по числу свойств:
Число свойств   | Число сущностей | Доля от выборки
--------------------------------------------------
              5 |               2 |   3.85%
              6 |              13 |  25.00%
              7 |              11 |  21.15%
              8 |              16 |  30.77%
              9 |               2 |   3.85%
             10 |               1 |   1.92%
             11 |               2 |   3.85%
             12 |               4 |   7.69%
             17 |               1 |   1.92%


## 1.3. Сущности с экстремальным числом свойств (минимум и максимум)

In [4]:
min_entities = [item for item in entity_props.items() if len(item[1]["props"]) == min_props]
print("Сущности с минимальным числом свойств (" + str(min_props) + "):")
for qid, data in min_entities:
    print(qid, "|", data["name"], "| свойства:", sorted(list(data["props"])))

max_entities = [item for item in entity_props.items() if len(item[1]["props"]) == max_props]
print("\nСущности с максимальным числом свойств (" + str(max_props) + "):")
for qid, data in max_entities:
    print(qid, "|", data["name"], "| свойства:", sorted(list(data["props"])))

Сущности с минимальным числом свойств (5):
Q120733208 | flexgen | свойства: ['P1324', 'P1547', 'P275', 'P31', 'P5568']
Q120550396 | pytorch-wavelets | свойства: ['P1324', 'P1547', 'P31', 'P348', 'P5568']

Сущности с максимальным числом свойств (17):
Q96631404 | PyKEEN | свойства: ['P1324', 'P1343', 'P1401', 'P1451', 'P1547', 'P178', 'P2037', 'P275', 'P277', 'P31', 'P348', 'P366', 'P4901', 'P5568', 'P6216', 'P6251', 'P856']


## 1.4. Полный список всех сущностей, ранжированных по количеству свойств

In [5]:
ranked_entities = sorted(entity_props.items(), key=lambda x: (len(x[1]["props"]), x[1]["name"]))

print(f"{'№':2} | {'QID':12} | {'Название':26} | {'Число':5} | Свойства")
print("-" * 80)
for idx, (qid, data) in enumerate(ranked_entities, 1):
    props_str = ", ".join(sorted(list(data["props"])))
    print(f"{idx:2} | {qid:12} | {data['name'][:26]:26} | {len(data['props']):5} | {props_str}")

№  | QID          | Название                   | Число | Свойства
--------------------------------------------------------------------------------
 1 | Q120733208   | flexgen                    |     5 | P1324, P1547, P275, P31, P5568
 2 | Q120550396   | pytorch-wavelets           |     5 | P1324, P1547, P31, P348, P5568
 3 | Q107382224   | bert-score                 |     6 | P1324, P1547, P275, P31, P348, P5568
 4 | Q120549792   | clean-fid                  |     6 | P1324, P1547, P275, P31, P5568, P856
 5 | Q107382238   | fairseq                    |     6 | P1324, P1547, P275, P31, P348, P5568
 6 | Q120733123   | fschat                     |     6 | P1324, P1547, P275, P31, P348, P5568
 7 | Q120549808   | gfpgan                     |     6 | P1324, P1547, P31, P348, P4162, P5568
 8 | Q120634941   | hummingbird-ml             |     6 | P1324, P1547, P275, P31, P348, P5568
 9 | Q120550360   | open-clip-torch            |     6 | P1324, P1547, P31, P348, P4162, P5568
10 | Q120550386  

# 2. Проверка домена критериям КР

(макс. 1 балл) Выбрать домен и зафиксировать его в общей таблице. Домены не должны пересекаться в рамках учебной группы. Требования к домену:
- домен содержит 30–60 сущностей одного основного класса в Wikidata;
- у большинства сущностей заполнены хотя бы 5 свойств, среди них даты и числа;
- хотя бы одно свойство имеет квалификатор времени (руководитель с датами, показатель с годом);
- по большинству сущностей есть текстовое описание (Википедия, официальный сайт, энциклопедия и т.п.).

## 2.1. домен содержит 30–60 сущностей одного основного класса в Wikidata

In [6]:
total_entities_count = len(entities)
is_size_valid = 30 <= total_entities_count <= 60

print("Основной класс в Wikidata: wd:Q29642950 (Python library)")
print("Число сущностей выборки:", total_entities_count)
print("Диапазон требования: от 30 до 60")
print("Критерий размера выборки выполнен:", is_size_valid)

print("\nПервые 5 сущностей выборки:")
for e in entities[:5]:
    print(e["qid"], "|", e["name"], "|", e["repo"])

Основной класс в Wikidata: wd:Q29642950 (Python library)
Число сущностей выборки: 52
Диапазон требования: от 30 до 60
Критерий размера выборки выполнен: True

Первые 5 сущностей выборки:
Q96631404 | PyKEEN | https://github.com/pykeen/pykeen
Q85847011 | PyTorch Lightning | https://github.com/Lightning-AI/lightning
Q116908797 | STonKGs | https://github.com/stonkgs/stonkgs
Q120550372 | Stanza | https://github.com/stanfordnlp/stanza
Q116908792 | accelerate | https://github.com/huggingface/accelerate


## 2.2. заполнены хотя бы 5 свойств

In [7]:
ge_5_entities = [qid for qid, data in entity_props.items() if len(data["props"]) >= 5]
ge_5_share = (len(ge_5_entities) / len(entity_props)) * 100

print("Всего сущностей проверено:", len(entity_props))
print("Минимальное число свойств у сущности:", min_props)
print("Сущностей с не менее чем 5 свойствами:", len(ge_5_entities))
print("Доля сущностей с не менее чем 5 свойствами:", str(round(ge_5_share, 2)) + "%")
print("Критерий большинства с не менее 5 свойствами выполнен:", ge_5_share >= 50.0)

Всего сущностей проверено: 52
Минимальное число свойств у сущности: 5
Сущностей с не менее чем 5 свойствами: 52
Доля сущностей с не менее чем 5 свойствами: 100.0%
Критерий большинства с не менее 5 свойствами выполнен: True


### 2.2.1. Есть даты

In [8]:
query_dates = """
SELECT ?item ?itemLabel ?version ?releaseDate WHERE {
  ?item wdt:P31 wd:Q29642950 ;
        wdt:P1547 wd:Q47509047 ;
        wdt:P1324 ?repo ;
        p:P348 ?statement .
  ?statement ps:P348 ?version ;
             pq:P577 ?releaseDate .
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en,ru,mul,da". }
}
ORDER BY DESC(?releaseDate)
"""

resp_dates = requests.get(url, params={"query": query_dates, "format": "json"}, headers=headers)
date_rows = resp_dates.json()["results"]["bindings"]
entities_with_dates = set(r["item"]["value"] for r in date_rows)

print("Всего утверждений с датами релиза (pq:P577):", len(date_rows))
print("Число сущностей, содержащих даты:", len(entities_with_dates))
print("Доля сущностей с датами:", str(round(len(entities_with_dates) / len(entity_props) * 100, 2)) + "%")
print("Тип данных свойства даты:", date_rows[0]["releaseDate"]["datatype"])
print("Критерий наличия дат выполнен:", len(date_rows) > 0)

print("\nПримеры зафиксированных дат:")
for r in date_rows[:5]:
    print(r["itemLabel"]["value"], "| версия:", r["version"]["value"], "| дата:", r["releaseDate"]["value"])

Всего утверждений с датами релиза (pq:P577): 1720
Число сущностей, содержащих даты: 47
Доля сущностей с датами: 90.38%
Тип данных свойства даты: http://www.w3.org/2001/XMLSchema#dateTime
Критерий наличия дат выполнен: True

Примеры зафиксированных дат:
open-clip-torch | версия: 4.2.2 | дата: 2026-11-10T00:00:00Z
pytorch-pretrained-bert | версия: 5.22.2 | дата: 2026-11-06T00:00:00Z
pytorch-transformers | версия: 5.22.2 | дата: 2026-11-06T00:00:00Z
open-clip-torch | версия: 4.2.1 | дата: 2026-11-06T00:00:00Z
timm | версия: 1.0.31 | дата: 2026-11-03T00:00:00Z


### 2.2.2. Есть числа

In [9]:
query_numbers = """
SELECT ?item ?itemLabel (COUNT(DISTINCT ?statement) AS ?versionCount) WHERE {
  ?item wdt:P31 wd:Q29642950 ;
        wdt:P1547 wd:Q47509047 ;
        wdt:P1324 ?repo ;
        p:P348 ?statement .
  ?statement ps:P348 ?version .
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en,ru,mul,da". }
}
GROUP BY ?item ?itemLabel
ORDER BY DESC(?versionCount)
"""

resp_numbers = requests.get(url, params={"query": query_numbers, "format": "json"}, headers=headers)
number_rows = resp_numbers.json()["results"]["bindings"]

numeric_records = []
for r in number_rows:
    numeric_records.append({
        "qid": r["item"]["value"].split("/")[-1],
        "name": r["itemLabel"]["value"],
        "count": int(r["versionCount"]["value"]),
        "datatype": r["versionCount"].get("datatype", "")
    })

num_values = [rec["count"] for rec in numeric_records]

print("Число сущностей с числовым свойством количества версий:", len(numeric_records))
print("Доля сущностей с числовыми значениями:", str(round(len(numeric_records) / len(entity_props) * 100, 2)) + "%")
print("Общее количество релизов (сумма чисел):", sum(num_values))
print("Минимальное число версий:", min(num_values))
print("Максимальное число версий:", max(num_values))
print("Среднее число версий на библиотеку:", round(sum(num_values) / len(num_values), 2))
print("Тип данных числового параметра:", numeric_records[0]["datatype"])
print("Критерий наличия чисел выполнен:", len(numeric_records) > 0)

print("\nПримеры числовых значений (топ-5 библиотек по числу версий):")
for rec in numeric_records[:5]:
    type_short = rec["datatype"].split("#")[-1]
    print(rec["qid"], "|", rec["name"], "| число версий:", rec["count"], "| xsd:" + type_short)

Число сущностей с числовым свойством количества версий: 47
Доля сущностей с числовыми значениями: 90.38%
Общее количество релизов (сумма чисел): 1616
Минимальное число версий: 1
Максимальное число версий: 242
Среднее число версий на библиотеку: 34.38
Тип данных числового параметра: http://www.w3.org/2001/XMLSchema#integer
Критерий наличия чисел выполнен: True

Примеры числовых значений (топ-5 библиотек по числу версий):
Q107385500 | pytorch-pretrained-bert | число версий: 242 | xsd:integer
Q107385836 | pytorch-transformers | число версий: 242 | xsd:integer
Q85847011 | PyTorch Lightning | число версий: 104 | xsd:integer
Q116908792 | accelerate | число версий: 67 | xsd:integer
Q107381756 | torchmetrics | число версий: 63 | xsd:integer


## 2.3. Свойство с квалификатором времени

In [10]:
print("Свойство, содержащее квалификатор времени: P348 (версия программы)")
print("Квалификатор времени: pq:P577 (дата публикации)")
print("Всего утверждений с данным квалификатором времени:", len(date_rows))
print("Число сущностей, у которых заполнен данный квалификатор времени:", len(entities_with_dates))
print("Доля сущностей с квалификатором времени:", str(round(len(entities_with_dates) / len(entity_props) * 100, 2)) + "%")
print("Критерий наличия квалификатора времени выполнен:", len(date_rows) > 0)

print("\nПервые 5 утверждений с квалификатором времени:")
for r in date_rows[:5]:
    print(r["itemLabel"]["value"], "| версия:", r["version"]["value"], "| дата публикации (pq:P577):", r["releaseDate"]["value"])

Свойство, содержащее квалификатор времени: P348 (версия программы)
Квалификатор времени: pq:P577 (дата публикации)
Всего утверждений с данным квалификатором времени: 1720
Число сущностей, у которых заполнен данный квалификатор времени: 47
Доля сущностей с квалификатором времени: 90.38%
Критерий наличия квалификатора времени выполнен: True

Первые 5 утверждений с квалификатором времени:
open-clip-torch | версия: 4.2.2 | дата публикации (pq:P577): 2026-11-10T00:00:00Z
pytorch-pretrained-bert | версия: 5.22.2 | дата публикации (pq:P577): 2026-11-06T00:00:00Z
pytorch-transformers | версия: 5.22.2 | дата публикации (pq:P577): 2026-11-06T00:00:00Z
open-clip-torch | версия: 4.2.1 | дата публикации (pq:P577): 2026-11-06T00:00:00Z
timm | версия: 1.0.31 | дата публикации (pq:P577): 2026-11-03T00:00:00Z


## 2.4. Текстовые описания

In [11]:
corpus_path = "data/corpus.jsonl"
with open(corpus_path, "r", encoding="utf-8") as f:
    corpus_docs = [json.loads(line) for line in f]

corpus_lengths = [len(d["text"]) for d in corpus_docs]

print("Файл текстового корпуса:", corpus_path)
print("Размер файла на диске в байтах:", os.path.getsize(corpus_path))
print("Число документов в корпусе:", len(corpus_docs))
print("Охват выборки текстовыми описаниями:", str(round(len(corpus_docs) / len(entity_props) * 100, 2)) + "%")
print("Минимальная длина текста в символах:", min(corpus_lengths))
print("Максимальная длина текста в символах:", max(corpus_lengths))
print("Средняя длина текста в символах:", round(sum(corpus_lengths) / len(corpus_lengths), 1))
print("Критерий наличия текстовых описаний выполнен:", len(corpus_docs) == len(entity_props))

print("\nПример первого документа корпуса:")
doc0 = corpus_docs[0]
print("QID:", doc0["id"])
print("Название:", doc0["name"])
print("Источник:", doc0["source_url"])
print("Длина текста:", len(doc0["text"]))
print("Фрагмент текста:\n", doc0["text"][:180])

Файл текстового корпуса: data/corpus.jsonl
Размер файла на диске в байтах: 216836
Число документов в корпусе: 52
Охват выборки текстовыми описаниями: 100.0%
Минимальная длина текста в символах: 2069
Максимальная длина текста в символах: 4000
Средняя длина текста в символах: 3885.9
Критерий наличия текстовых описаний выполнен: True

Пример первого документа корпуса:
QID: Q96631404
Название: PyKEEN
Источник: https://raw.githubusercontent.com/pykeen/pykeen/master/README.md
Длина текста: 4000
Фрагмент текста:
 <p align="center">
  <img src="docs/source/logo.png" height="150">
</p>

<h1 align="center">
  PyKEEN
</h1>

<p align="center">
  <a href="https://github.com/pykeen/pykeen/actions/


# 3. Обязательная статистика

После выбора домена вы должны проверить его на требования, описанные выше. Для этого требуется SPARQL-запросом к Wikidata посчитать сущности основного класса и заполненность их свойств. В отчете указывается:
- статистика по основным свойствам (доля сущностей, у которых это свойство заполнено)
- набор свойств, имеющих квалификатор времени
- для каждой сущности – ссылка на источник текстовой информации

## 3.1. статистика по основным свойствам

In [12]:
prop_labels = {
    "P1324": "Репозиторий с исходным кодом",
    "P31": "Экземпляр класса",
    "P1547": "Зависимость от программного обеспечения",
    "P5568": "Тема GitHub",
    "P275": "Лицензия",
    "P348": "Версия программы",
    "P856": "Официальный сайт",
    "P4162": "Проект PyPI",
    "P6216": "Статус авторских прав",
    "P277": "Язык программирования",
    "P178": "Разработчик",
    "P1343": "Описано в источнике",
    "P1401": "Баг-трекер",
    "P366": "Использование",
    "P2078": "Руководство пользователя"
}

prop_counts = {}
for qid, data in entity_props.items():
    for pid in data["props"]:
        prop_counts[pid] = prop_counts.get(pid, 0) + 1

sorted_prop_counts = sorted(prop_counts.items(), key=lambda x: x[1], reverse=True)

print(f"{'Свойство':8} | {'Название':42} | {'Заполнено':9} | Доля от выборки")
print("-" * 75)
for pid, filled in sorted_prop_counts:
    label = prop_labels.get(pid, pid)
    ratio = (filled / len(entity_props)) * 100
    print(f"{pid:8} | {label:42} | {filled:2}/{len(entity_props)}    | {ratio:6.2f}%")

Свойство | Название                                   | Заполнено | Доля от выборки
---------------------------------------------------------------------------
P1324    | Репозиторий с исходным кодом               | 52/52    | 100.00%
P5568    | Тема GitHub                                | 52/52    | 100.00%
P1547    | Зависимость от программного обеспечения    | 52/52    | 100.00%
P31      | Экземпляр класса                           | 52/52    | 100.00%
P275     | Лицензия                                   | 49/52    |  94.23%
P348     | Версия программы                           | 47/52    |  90.38%
P856     | Официальный сайт                           | 33/52    |  63.46%
P4162    | Проект PyPI                                | 27/52    |  51.92%
P6216    | Статус авторских прав                      |  9/52    |  17.31%
P277     | Язык программирования                      |  8/52    |  15.38%
P178     | Разработчик                                |  7/52    |  13.46%
P1343    | Опис

## 3.2. набор свойств, имеющих квалификатор времени

In [13]:
qual_props_info = [
    {
        "prop_id": "P348",
        "prop_label": "версия программы",
        "qual_id": "pq:P577",
        "qual_label": "дата публикации",
        "datatype": "xsd:dateTime",
        "statements_count": len(date_rows),
        "entities_count": len(entities_with_dates),
        "share": round(len(entities_with_dates) / len(entity_props) * 100, 2)
    }
]

print("Набор свойств, имеющих квалификатор времени в домене:")
for item in qual_props_info:
    print("Основное свойство:", item["prop_id"], "(" + item["prop_label"] + ")")
    print("Квалификатор времени:", item["qual_id"], "(" + item["qual_label"] + ")")
    print("Тип данных квалификатора:", item["datatype"])
    print("Число утверждений с квалификатором:", item["statements_count"])
    print("Охват сущностей:", str(item["entities_count"]) + "/" + str(len(entity_props)), "(" + str(item["share"]) + "%)")

Набор свойств, имеющих квалификатор времени в домене:
Основное свойство: P348 (версия программы)
Квалификатор времени: pq:P577 (дата публикации)
Тип данных квалификатора: xsd:dateTime
Число утверждений с квалификатором: 1720
Охват сущностей: 47/52 (90.38%)


## 3.3. ссылка на источник текстовой информации

In [14]:
print(f"{'№':2} | {'QID':12} | {'Название библиотеки':26} | Ссылка на текстовый источник (README.md)")
print("-" * 105)
for idx, e in enumerate(entities, 1):
    raw_readme_url = e["repo"].replace("https://github.com/", "https://raw.githubusercontent.com/") + "/master/README.md"
    print(f"{idx:2} | {e['qid']:12} | {e['name'][:26]:26} | {raw_readme_url}")

№  | QID          | Название библиотеки        | Ссылка на текстовый источник (README.md)
---------------------------------------------------------------------------------------------------------
 1 | Q96631404    | PyKEEN                     | https://raw.githubusercontent.com/pykeen/pykeen/master/README.md
 2 | Q85847011    | PyTorch Lightning          | https://raw.githubusercontent.com/Lightning-AI/lightning/master/README.md
 3 | Q116908797   | STonKGs                    | https://raw.githubusercontent.com/stonkgs/stonkgs/master/README.md
 4 | Q120550372   | Stanza                     | https://raw.githubusercontent.com/stanfordnlp/stanza/master/README.md
 5 | Q116908792   | accelerate                 | https://raw.githubusercontent.com/huggingface/accelerate/master/README.md
 6 | Q107382302   | allennlp                   | https://raw.githubusercontent.com/allenai/allennlp/master/README.md
 7 | Q107382376   | attacut                    | https://raw.githubusercontent.com/PyThaiNLP